In [46]:
import mediapipe as mp
print(mp.__version__)  # Should print 0.9.3

0.10.35


In [47]:
import cv2
import mediapipe as mp

# ── Setup ──────────────────────────────────────────────
latest_result = None

def on_result(result, output_image, timestamp_ms):
    global latest_result
    latest_result = result

options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=mp.tasks.vision.RunningMode.LIVE_STREAM,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    result_callback=on_result
)

CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),         # Thumb
    (0,5),(5,6),(6,7),(7,8),         # Index
    (0,9),(9,10),(10,11),(11,12),    # Middle
    (0,13),(13,14),(14,15),(15,16),  # Ring
    (0,17),(17,18),(18,19),(19,20),  # Pinky
    (5,9),(9,13),(13,17)             # Palm
]

# ── Main Loop ──────────────────────────────────────────
cap = cv2.VideoCapture(0)
timestamp = 0

with mp.tasks.vision.HandLandmarker.create_from_options(options) as landmarker:
    while True:
        success, img = cap.read()
        if not success:
            break

        # Send frame to detector
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        landmarker.detect_async(mp_img, timestamp)
        timestamp += 1

        # Draw results
        if latest_result and latest_result.hand_landmarks:
            h, w, _ = img.shape
            for hand in latest_result.hand_landmarks:
                pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand]

                # Connections
                for a, b in CONNECTIONS:
                    cv2.line(img, pts[a], pts[b], (0, 255, 0), 2)

                # Landmarks
                for i, (cx, cy) in enumerate(pts):
                    cv2.circle(img, (cx, cy), 6, (255, 0, 255), -1)
                    cv2.putText(img, str(i), (cx+5, cy+5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255,255,255), 1)

        # FPS display
        cv2.putText(img, "Press Q to quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        cv2.imshow("Hand Tracking", img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [48]:
pip install pycaw screen-brightness-control comtypes

Note: you may need to restart the kernel to use updated packages.


In [49]:
import cv2
import mediapipe as mp
import numpy as np
import math
import os
import screen_brightness_control as sbc
from ctypes import cast, POINTER
from comtypes import CLSCTX_ALL
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume

# ── Audio Setup ────────────────────────────────────────
# ── Audio Setup ────────────────────────────────────────
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume
from ctypes import cast, POINTER
from comtypes import CLSCTX_ALL

# ✅ New correct way
devices = AudioUtilities.GetSpeakers()
interface = devices.Activate(
    IAudioEndpointVolume._iid_, CLSCTX_ALL, None
)

# ── MediaPipe Setup ────────────────────────────────────
latest_result = None

def on_result(result, output_image, timestamp_ms):
    global latest_result
    latest_result = result

model_path = os.path.abspath("hand_landmarker.task")

options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=model_path),
    running_mode=mp.tasks.vision.RunningMode.LIVE_STREAM,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    result_callback=on_result
)

CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17)
]

def get_distance(p1, p2):
    return math.hypot(p2[0] - p1[0], p2[1] - p1[1])

def draw_bar(img, x, y, w, h, pct, color, label):
    """Draw a vertical bar indicator"""
    cv2.rectangle(img, (x, y), (x+w, y+h), (50,50,50), -1)
    filled = int(h * pct / 100)
    cv2.rectangle(img, (x, y + h - filled), (x+w, y+h), color, -1)
    cv2.rectangle(img, (x, y), (x+w, y+h), (200,200,200), 2)
    cv2.putText(img, f"{label}: {int(pct)}%", (x - 10, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

# ── Main Loop ──────────────────────────────────────────
cap = cv2.VideoCapture(0)
timestamp = 0

with mp.tasks.vision.HandLandmarker.create_from_options(options) as landmarker:
    while True:
        success, img = cap.read()
        if not success:
            break

        img = cv2.flip(img, 1)  # Mirror view
        h, w, _ = img.shape

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        landmarker.detect_async(mp_img, timestamp)
        timestamp += 1

        vol_pct = 0
        bright_pct = 0

        if latest_result and latest_result.hand_landmarks:
            hands_data = latest_result.hand_landmarks
            handedness = latest_result.handedness if latest_result.handedness else []

            for i, hand in enumerate(hands_data):
                pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand]

                # Draw connections
                for a, b in CONNECTIONS:
                    cv2.line(img, pts[a], pts[b], (0, 255, 0), 2)

                # Draw landmarks
                for cx, cy in pts:
                    cv2.circle(img, (cx, cy), 6, (255, 0, 255), -1)

                # Thumb tip = 4, Index tip = 8
                thumb  = pts[4]
                index  = pts[8]
                dist   = get_distance(thumb, index)
                mid    = ((thumb[0]+index[0])//2, (thumb[1]+index[1])//2)

                # Draw pinch line
                cv2.line(img, thumb, index, (255, 255, 0), 2)
                cv2.circle(img, mid, 8, (0, 255, 255), -1)

                # Map distance → 0–100%
                pct = np.interp(dist, [20, 200], [0, 100])

                # Determine which hand controls what
                label = ""
                if handedness and i < len(handedness):
                    label = handedness[i][0].display_name  # "Left" or "Right"

                if label == "Right":
                    # RIGHT hand → Volume
                    vol_level = np.interp(pct, [0, 100], [min_vol, max_vol])
                    volume.SetMasterVolumeLevel(vol_level, None)
                    vol_pct = pct
                    cv2.putText(img, f"VOL: {int(pct)}%", (pts[8][0]-20, pts[8][1]-20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 255), 2)

                elif label == "Left":
                    # LEFT hand → Brightness
                    bright_level = int(np.interp(pct, [0, 100], [0, 100]))
                    try:
                        sbc.set_brightness(bright_level)
                    except Exception:
                        pass
                    bright_pct = pct
                    cv2.putText(img, f"BRIGHT: {int(pct)}%", (pts[8][0]-20, pts[8][1]-20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 150), 2)

            # Bars
            try:
                vol_pct = np.interp(volume.GetMasterVolumeLevel(), [min_vol, max_vol], [0, 100])
            except:
                pass
            try:
                bright_pct = sbc.get_brightness()[0]
            except:
                pass

        # Draw bars on right side
        draw_bar(img, w-80, 100, 30, 300, vol_pct,    (0, 200, 255), "VOL")
        draw_bar(img, w-140, 100, 30, 300, bright_pct, (0, 255, 150), "BRIT")

        cv2.putText(img, "RIGHT hand = Volume | LEFT hand = Brightness",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)
        cv2.putText(img, "Pinch fingers to control | Q to quit",
                    (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)

        cv2.imshow("Hand Control", img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


AttributeError: 'AudioDevice' object has no attribute 'Activate'

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os

latest_result = None
lines = []          # stores all completed lines
current_line = []   # points in current stroke

def on_result(result, output_image, timestamp_ms):
    global latest_result
    latest_result = result

model_path = os.path.abspath("hand_landmarker.task")
options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=model_path),
    running_mode=mp.tasks.vision.RunningMode.LIVE_STREAM,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    result_callback=on_result
)

CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17)
]

def fingers_up(pts):
    tips = [8, 12, 16, 20]
    pips = [6, 10, 14, 18]
    return [1 if pts[t][1] < pts[p][1] else 0 for t, p in zip(tips, pips)]

cap = cv2.VideoCapture(0)
timestamp = 0

with mp.tasks.vision.HandLandmarker.create_from_options(options) as landmarker:
    while True:
        success, img = cap.read()
        if not success:
            break

        img = cv2.flip(img, 1)
        h, w, _ = img.shape

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        landmarker.detect_async(mp_img, timestamp)
        timestamp += 1

        mode = "IDLE"

        if latest_result and latest_result.hand_landmarks:
            for hand in latest_result.hand_landmarks:
                pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand]

                # Draw skeleton
                for a, b in CONNECTIONS:
                    cv2.line(img, pts[a], pts[b], (0, 255, 0), 1)
                for cx, cy in pts:
                    cv2.circle(img, (cx, cy), 4, (255, 0, 255), -1)

                f = fingers_up(pts)
                index_tip = pts[8]

                if f == [1, 0, 0, 0]:
                    # ☝ ONE finger → DRAW
                    mode = "DRAWING"
                    current_line.append(index_tip)
                    cv2.circle(img, index_tip, 8, (0, 0, 255), -1)

                elif f == [1, 1, 0, 0]:
                    # ✌ TWO fingers → STOP & save line
                    mode = "STOP"
                    if len(current_line) > 1:
                        lines.append(current_line.copy())
                    current_line.clear()

                elif f == [1, 1, 1, 1]:
                    # 🖐 FOUR fingers → CLEAR all
                    mode = "CLEAR"
                    lines.clear()
                    current_line.clear()

        # Draw all saved lines
        for line in lines:
            for i in range(1, len(line)):
                cv2.line(img, line[i-1], line[i], (255, 100, 0), 3)

        # Draw current active line
        for i in range(1, len(current_line)):
            cv2.line(img, current_line[i-1], current_line[i], (0, 100, 255), 3)

        # Mode badge
        badge_color = {
            "DRAWING": (0, 0, 255),
            "STOP":    (0, 200, 0),
            "CLEAR":   (0, 200, 200),
            "IDLE":    (80, 80, 80)
        }[mode]
        cv2.rectangle(img, (w-160, 10), (w-10, 50), badge_color, -1)
        cv2.putText(img, mode, (w-150, 38),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

        # Instructions
        cv2.putText(img, "1 finger=Draw  2=Stop  4=Clear  Q=Quit",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,255), 1)

        cv2.imshow("Air Line Drawing", img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()